# Deutsch--Jozsa y Bernstein--Vazirani

Este notebook es el material complementario ejecutable para el módulo. Contiene las derivaciones matemáticas completas, ejemplos desarrollados paso a paso y simulaciones en Qiskit. La presentación principal prioriza intuición e interpretación; aquí se conservan los detalles algebraicos y computacionales para estudio individual, verificación formal y práctica en JupyterLab, Anaconda o Google Colab.

La idea unificadora es que el oráculo transforma información sobre una función booleana en fases relativas; Hadamard convierte esas fases en una salida medible. Deutsch--Jozsa usa este mecanismo para decidir si una función prometida es constante o balanceada. Bernstein--Vazirani lo usa para recuperar una cadena secreta $s$ que define una función lineal booleana.

## Objetivos de aprendizaje

Al finalizar este notebook deberías poder:

1. Formular el problema de Deutsch--Jozsa para $f:\{0,1\}^n\to\{0,1\}$ bajo la promesa constante/balanceada.
2. Explicar por qué el modelo clásico determinista requiere $2^{n-1}+1$ consultas en el peor caso para Deutsch--Jozsa.
3. Derivar la identidad
   $$
   H^{\otimes n}|x\rangle=\frac{1}{\sqrt{2^n}}\sum_{z\in\{0,1\}^n}(-1)^{x\cdot z}|z\rangle.
   $$
4. Demostrar el *phase kickback*
   $$
   U_f|x\rangle|-\rangle=(-1)^{f(x)}|x\rangle|-\rangle.
   $$
5. Derivar la amplitud final de Deutsch--Jozsa.
6. Formular Bernstein--Vazirani como recuperación de $s$ en $f_s(x)=x\cdot s\pmod 2$.
7. Demostrar que Bernstein--Vazirani devuelve $|s\rangle$ con probabilidad $1$ en el modelo ideal.
8. Implementar oráculos y circuitos completos en Qiskit.

## 1. Fundamentos matemáticos

Usaremos cadenas binarias $x=x_1x_2\cdots x_n\in\{0,1\}^n$ para etiquetar estados base:

$$
|x\rangle=|x_1\rangle\otimes |x_2\rangle\otimes\cdots\otimes |x_n\rangle.
$$

El producto interno módulo $2$ se define como

$$
x\cdot z=x_1z_1\oplus x_2z_2\oplus\cdots\oplus x_nz_n.
$$

Este producto no mide ángulos geométricos usuales; cuenta la paridad de las posiciones en las que ambos vectores tienen un $1$.

### 1.1 Hadamard multidimensional

Para un qubit,

$$
H|0\rangle=\frac{|0\rangle+|1\rangle}{\sqrt 2},\qquad
H|1\rangle=\frac{|0\rangle-|1\rangle}{\sqrt 2}.
$$

Ambos casos se escriben de forma compacta como

$$
H|x_i\rangle=\frac{1}{\sqrt2}\sum_{z_i\in\{0,1\}}(-1)^{x_i z_i}|z_i\rangle.
$$

Si $x_i=0$, el exponente $x_i z_i$ siempre es $0$, por lo que ambos signos son positivos. Si $x_i=1$, el término con $z_i=1$ recibe signo negativo.

Para $n$ qubits:

$$
\begin{aligned}
H^{\otimes n}|x\rangle
&=\bigotimes_{i=1}^{n}H|x_i\rangle\\
&=\bigotimes_{i=1}^{n}\left(\frac{1}{\sqrt2}\sum_{z_i\in\{0,1\}}(-1)^{x_i z_i}|z_i\rangle\right)\\
&=\frac{1}{\sqrt{2^n}}\sum_{z_1,\ldots,z_n\in\{0,1\}}(-1)^{x_1z_1+\cdots+x_nz_n}|z_1\cdots z_n\rangle\\
&=\frac{1}{\sqrt{2^n}}\sum_{z\in\{0,1\}^n}(-1)^{x\cdot z}|z\rangle.
\end{aligned}
$$

Interpretación: $H^{\otimes n}$ convierte una etiqueta $x$ en un patrón de signos sobre todas las etiquetas $z$.

### 1.2 Oráculos reversibles y *phase kickback*

Un oráculo booleano reversible se define por

$$
U_f|x\rangle|y\rangle=|x\rangle|y\oplus f(x)\rangle.
$$

El segundo registro es necesario porque la transformación $|x\rangle\mapsto |f(x)\rangle$ puede no ser reversible: muchas entradas distintas podrían tener el mismo valor de salida.

El estado auxiliar clave es

$$
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt2}.
$$

Demostración completa del *phase kickback*:

$$
\begin{aligned}
U_f|x\rangle|-\rangle
&=U_f|x\rangle\left(\frac{|0\rangle-|1\rangle}{\sqrt2}\right)\\
&=\frac{1}{\sqrt2}\left(U_f|x\rangle|0\rangle-U_f|x\rangle|1\rangle\right)\\
&=\frac{1}{\sqrt2}\left(|x\rangle|0\oplus f(x)\rangle-|x\rangle|1\oplus f(x)\rangle\right).
\end{aligned}
$$

Si $f(x)=0$:

$$
U_f|x\rangle|-\rangle=\frac{|x\rangle|0\rangle-|x\rangle|1\rangle}{\sqrt2}=|x\rangle|-\rangle.
$$

Si $f(x)=1$:

$$
U_f|x\rangle|-\rangle=\frac{|x\rangle|1\rangle-|x\rangle|0\rangle}{\sqrt2}=-|x\rangle|-\rangle.
$$

Por tanto,

$$
U_f|x\rangle|-\rangle=(-1)^{f(x)}|x\rangle|-\rangle.
$$

## 2. Configuración de Qiskit

El siguiente bloque instala Qiskit y Qiskit Aer si no están disponibles. En Google Colab esto normalmente descarga los paquetes. En JupyterLab o Anaconda, si ya están instalados, el bloque solo los importa.

In [ ]:
# Instalación/importación robusta para JupyterLab, Anaconda y Google Colab.
import sys
import subprocess

try:
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "qiskit", "qiskit-aer"])
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator

from collections import Counter

In [ ]:
def run_counts(circuit, shots=1024):
    """Ejecuta un circuito medido usando AerSimulator y devuelve conteos."""
    simulator = AerSimulator()
    compiled = transpile(circuit, simulator)
    result = simulator.run(compiled, shots=shots).result()
    return result.get_counts(compiled)


def print_counts(title, counts):
    """Imprime conteos ordenados para facilitar lectura."""
    print(title)
    for bitstring, count in sorted(counts.items()):
        print(f"  {bitstring}: {count}")

## 3. Deutsch--Jozsa

### 3.1 Problema

Dada una función

$$
f:\{0,1\}^n\to\{0,1\},
$$

se promete que ocurre exactamente uno de estos casos:

1. **Constante:** $f(x)=c$ para todo $x$.
2. **Balanceada:** $f(x)=0$ en exactamente $2^{n-1}$ entradas y $f(x)=1$ en exactamente $2^{n-1}$ entradas.

El objetivo es decidir cuál caso ocurre.

La pregunta no es: “¿cuáles son todos los valores de $f$?”. La pregunta es: “¿qué propiedad global tiene $f$?”.

### 3.2 Costo clásico determinista

Un algoritmo clásico determinista que requiere certeza debe considerar el peor caso.

Si consulta $2^{n-1}$ entradas y todas producen el mismo valor, todavía existen dos posibilidades compatibles:

- la función es constante;
- la función es balanceada y todas las entradas consultadas pertenecen a la mitad con el mismo valor.

Por eso se necesita una consulta adicional:

$$
2^{n-1}+1.
$$

El algoritmo cuántico usa una sola consulta porque no busca un valor opuesto por inspección; busca cancelación de amplitudes.

### 3.3 Derivación del algoritmo Deutsch--Jozsa

Estado inicial:

$$
|\psi_0\rangle=|0^n\rangle|1\rangle.
$$

Aplicamos Hadamard al registro de entrada y al auxiliar:

$$
|\psi_1\rangle=\left(\frac{1}{\sqrt{2^n}}\sum_{x\in\{0,1\}^n}|x\rangle\right)|-\rangle.
$$

Aplicamos el oráculo y usamos *phase kickback*:

$$
|\psi_2\rangle=\frac{1}{\sqrt{2^n}}\sum_x(-1)^{f(x)}|x\rangle|-\rangle.
$$

El auxiliar ya no contiene la respuesta útil; permanece factorizado como $|-\rangle$.

Aplicamos $H^{\otimes n}$ al registro de entrada:

$$
\begin{aligned}
|\psi_3\rangle
&=\frac{1}{\sqrt{2^n}}\sum_x(-1)^{f(x)}H^{\otimes n}|x\rangle|-\rangle\\
&=\frac{1}{\sqrt{2^n}}\sum_x(-1)^{f(x)}
\left(\frac{1}{\sqrt{2^n}}\sum_z(-1)^{x\cdot z}|z\rangle\right)|-\rangle\\
&=\sum_z\left[\frac{1}{2^n}\sum_x(-1)^{f(x)+x\cdot z}\right]|z\rangle|-\rangle.
\end{aligned}
$$

La amplitud final de $|z\rangle$ es

$$
\alpha_z=\frac{1}{2^n}\sum_x(-1)^{f(x)+x\cdot z}.
$$

### 3.4 Interpretación de la amplitud

La fórmula

$$
\alpha_z=\frac{1}{2^n}\sum_x(-1)^{f(x)+x\cdot z}
$$

puede leerse así:

- cada entrada $x$ aporta un voto $+1$ o $-1$;
- el valor $f(x)$ decide el signo escrito por el oráculo;
- el término $x\cdot z$ decide el signo del detector Hadamard;
- la amplitud mide correlación entre ambos patrones.

Para $z=0^n$:

$$
\alpha_{0^n}=\frac{1}{2^n}\sum_x(-1)^{f(x)}.
$$

Si $f$ es constante, todos los signos son iguales y $|\alpha_{0^n}|=1$. Si $f$ es balanceada, la mitad de signos son positivos y la mitad negativos, por lo que $\alpha_{0^n}=0$.

### 3.5 Regla de decisión

En el modelo ideal:

$$
\boxed{\text{medir }0^n\Rightarrow \text{función constante}}
$$

$$
\boxed{\text{medir algo distinto de }0^n\Rightarrow \text{función balanceada}}
$$

Esta regla solo es válida bajo la promesa constante/balanceada.

### 3.6 Implementación de Deutsch--Jozsa en Qiskit

Para mantener coherencia con la notación del texto, usaremos la convención: la cadena mostrada por Qiskit debe leerse en el mismo orden conceptual que los qubits de entrada. Por eso medimos el qubit $i$ en el bit clásico $n-1-i$.

In [ ]:
def constant_oracle(n, c=0):
    """Oráculo para f(x)=c. El qubit n es el auxiliar/salida."""
    qc = QuantumCircuit(n + 1, name=f"const_{c}")
    if c == 1:
        qc.x(n)
    return qc


def linear_balanced_oracle(mask):
    """Oráculo para f(x)=x·mask mod 2. Si mask no es cero, es balanceado."""
    n = len(mask)
    qc = QuantumCircuit(n + 1, name=f"balanced_{mask}")
    for i, bit in enumerate(mask):
        if bit == "1":
            qc.cx(i, n)
    return qc


def deutsch_jozsa_circuit(n, oracle):
    """Circuito completo de Deutsch--Jozsa."""
    qc = QuantumCircuit(n + 1, n)

    # Prepara el auxiliar en |->.
    qc.x(n)
    qc.h(n)

    # Superposición uniforme en el registro de entrada.
    qc.h(range(n))

    # Consulta al oráculo.
    qc.compose(oracle, inplace=True)

    # Interferencia final.
    qc.h(range(n))

    # Medición con orden legible de izquierda a derecha.
    for i in range(n):
        qc.measure(i, n - 1 - i)
    return qc

In [ ]:
# Ejemplo 1: función constante cero.
n = 4
oracle_c0 = constant_oracle(n, c=0)
circuit_c0 = deutsch_jozsa_circuit(n, oracle_c0)
counts_c0 = run_counts(circuit_c0)
print_counts("Deutsch--Jozsa con función constante cero", counts_c0)

In [ ]:
# Ejemplo 2: función balanceada f(x)=x_1 xor x_3, representada por mask="1010".
mask = "1010"
oracle_bal = linear_balanced_oracle(mask)
circuit_bal = deutsch_jozsa_circuit(len(mask), oracle_bal)
counts_bal = run_counts(circuit_bal)
print_counts(f"Deutsch--Jozsa con función balanceada mask={mask}", counts_bal)

Interpretación de los resultados:

- En el caso constante, los conteos deben concentrarse en `0000`.
- En el caso balanceado lineal, los conteos deben concentrarse en una cadena distinta de `0000`.

El punto conceptual no es que hayamos recuperado toda la tabla de verdad, sino que la interferencia final detectó si el paisaje de fase tenía componente uniforme.

### 3.7 Ejercicio guiado Deutsch--Jozsa

**Enunciado.** Para $n=2$, considere

$$
f(00)=0,
\quad f(01)=1,
\quad f(10)=1,
\quad f(11)=0.
$$

1. Clasifica la función.
2. Predice si el resultado puede ser $00$.
3. Explica qué significa físicamente la predicción.

**Solución.** Hay dos ceros y dos unos; por tanto la función es balanceada. En Deutsch--Jozsa, las funciones balanceadas cancelan la amplitud de $|00\rangle$. Por tanto, el resultado ideal no puede ser $00$.

Físicamente, los caminos con signo positivo y negativo llegan al detector uniforme con amplitudes opuestas y se cancelan.

## 4. Bernstein--Vazirani

### 4.1 Problema

Existe una cadena secreta $s\in\{0,1\}^n$ tal que

$$
f_s(x)=x\cdot s\pmod 2.
$$

El objetivo es recuperar $s$.

Si $s_i=1$, el bit $x_i$ participa en la paridad. Si $s_i=0$, el bit $x_i$ no afecta el valor de la función.

### 4.2 Estrategia clásica

Sea $e_i$ el vector unitario con un único $1$ en la posición $i$. Entonces

$$
f_s(e_i)=e_i\cdot s=s_i.
$$

Por tanto, un algoritmo clásico determinista puede recuperar $s$ con $n$ consultas:

$$
f_s(e_1)=s_1,\quad f_s(e_2)=s_2,\quad\ldots,\quad f_s(e_n)=s_n.
$$

El algoritmo cuántico recupera toda la cadena con una sola consulta, porque consulta en una base que convierte la regla lineal completa en un patrón de fase.

### 4.3 Derivación cuántica

Preparamos

$$
\frac{1}{\sqrt{2^n}}\sum_x|x\rangle|-\rangle.
$$

Aplicamos el oráculo de $f_s$:

$$
\frac{1}{\sqrt{2^n}}\sum_x|x\rangle|-\rangle
\longmapsto
\frac{1}{\sqrt{2^n}}\sum_x(-1)^{x\cdot s}|x\rangle|-\rangle.
$$

Pero por la identidad de Hadamard:

$$
H^{\otimes n}|s\rangle=\frac{1}{\sqrt{2^n}}\sum_x(-1)^{s\cdot x}|x\rangle.
$$

Como $s\cdot x=x\cdot s$, el registro de entrada está en $H^{\otimes n}|s\rangle$.

Aplicamos Hadamard otra vez:

$$
H^{\otimes n}H^{\otimes n}|s\rangle=|s\rangle.
$$

La medición devuelve $s$ con probabilidad $1$ en el modelo ideal.

### 4.4 Implementación de Bernstein--Vazirani en Qiskit

In [ ]:
def bv_oracle(s):
    """Oráculo de Bernstein--Vazirani para f_s(x)=x·s mod 2."""
    n = len(s)
    qc = QuantumCircuit(n + 1, name=f"bv_{s}")
    for i, bit in enumerate(s):
        if bit == "1":
            qc.cx(i, n)
    return qc


def bernstein_vazirani_circuit(s):
    """Circuito completo de Bernstein--Vazirani."""
    n = len(s)
    qc = QuantumCircuit(n + 1, n)

    # Auxiliar en |->.
    qc.x(n)
    qc.h(n)

    # Entrada uniforme.
    qc.h(range(n))

    # Oráculo lineal.
    qc.compose(bv_oracle(s), inplace=True)

    # Decodificación por Hadamard.
    qc.h(range(n))

    # Medición con orden conceptual legible.
    for i in range(n):
        qc.measure(i, n - 1 - i)
    return qc

In [ ]:
# Ejemplo BV: recuperar s = "0110".
s = "0110"
bv_circ = bernstein_vazirani_circuit(s)
bv_counts = run_counts(bv_circ)
print_counts(f"Bernstein--Vazirani con s={s}", bv_counts)

Interpretación: si los conteos se concentran en `0110`, el simulador confirma que la interferencia final reconstruyó la cadena secreta. La salida no es un valor $f(x)$, sino el parámetro estructural que define la función.

### 4.5 Ejercicio guiado Bernstein--Vazirani

**Enunciado.** Supón que una función prometida satisface

$$
f(100)=1,
\qquad f(010)=1,
\qquad f(001)=0.
$$

Si $f(x)=x\cdot s$, recupera $s$.

**Desarrollo.** Cada consulta a un vector unitario revela un bit de $s$:

$$
f(100)=s_1=1,
$$

$$
f(010)=s_2=1,
$$

$$
f(001)=s_3=0.
$$

Por tanto,

$$
s=110.
$$

**Interpretación.** La función depende de los dos primeros bits y no depende del tercero. El algoritmo cuántico obtendría `110` en una sola consulta coherente.

## 5. Comparación conceptual

| Aspecto | Deutsch--Jozsa | Bernstein--Vazirani |
|---|---|---|
| Promesa | constante o balanceada | lineal $x\cdot s$ |
| Pregunta | ¿qué categoria tiene $f$? | ¿cuál es $s$? |
| Salida ideal | $0^n$ o distinto de $0^n$ | $s$ |
| Papel del oráculo | crear un paisaje de fase | crear el paisaje de fase de una máscara lineal |
| Papel de Hadamard | detectar componente uniforme | invertir el cambio de base y recuperar $s$ |

La lección principal es que el circuito no debe interpretarse aislado. La promesa del problema determina qué significa la medición.

## 6. Ejercicios propuestos

### Ejercicio 1: Deutsch--Jozsa conceptual

Para $n=3$, una función prometida tiene cuatro entradas con salida $0$ y cuatro con salida $1$.

1. ¿Qué tipo de función es?
2. ¿Puede el algoritmo devolver `000` en el modelo ideal?
3. ¿Qué fenómeno físico explica tu respuesta?

**Respuesta esperada.** Es balanceada. No debe devolver `000`. El fenómeno físico es interferencia destructiva en la amplitud del componente uniforme.

### Ejercicio 2: Bernstein--Vazirani conceptual

Si el resultado ideal del algoritmo BV es `1001`, explica qué bits de entrada afectan la función.

**Respuesta esperada.** Participan el primer y el cuarto bit; los bits intermedios no afectan la paridad.

### Ejercicio 3: condición alterada

¿Qué falla si en lugar de preparar el auxiliar en $|-\rangle$ se prepara en $|+\rangle$?

**Respuesta esperada.** Como $X|+\rangle=|+\rangle$, el oráculo no produce el signo dependiente de $f(x)$. Sin el paisaje de fase, la interferencia final no codifica la propiedad buscada.

In [ ]:
# Verificación programática de varios casos BV.
for secret in ["0000", "1001", "1111", "0101"]:
    circ = bernstein_vazirani_circuit(secret)
    counts = run_counts(circ, shots=256)
    print_counts(f"s={secret}", counts)
    print()

## 7. Resumen final

Deutsch--Jozsa y Bernstein--Vazirani comparten una estructura:

$$
|0^n\rangle|1\rangle
\longrightarrow
\frac{1}{\sqrt{2^n}}\sum_x|x\rangle|-\rangle
\longrightarrow
\frac{1}{\sqrt{2^n}}\sum_x(-1)^{f(x)}|x\rangle|-\rangle
\longrightarrow
\text{medición informativa}.
$$

La diferencia está en la promesa:

- En Deutsch--Jozsa, la promesa permite decidir constante frente a balanceada.
- En Bernstein--Vazirani, la promesa lineal permite recuperar la cadena secreta $s$.

El concepto que debes conservar es: los oráculos no solo devuelven valores; usados coherentemente, pueden escribir fases. La interferencia convierte esas fases en información computacional observable.